# PageRank Implementation in Numpy

In [114]:
import numpy as np
import matplotlib.pyplot as plt

## Mini- WEB - Based Data Structure

Firistly, we will model of a small modelled web of three Pages: A,B and C. We will build a simple 3-page cyce (A $\rightarrow$ B $\rightarrow$ C $\rightarrow$ A) and an array that reprsentes the link structure (Adjacency Matirces). This will be our 3-page test graph to verify the PageRank logic later. 

In [53]:
#3-Page cyclic Web and .
# Build Adjacency matrix: A[i,j] = 1 if page j links to page i
A_test = np.array([
    [0,0,1], # Row (0): Page A is connected from page C 
    [1,0,0], # Row (1): Page B is connected from page A
    [0,1,0] # Row (2): Page C is connected from page B
])

A_test = np.array(A_test, dtype = float)

print("3-page Web") 
print("\nAdjancency Matrix Visualization:")
print("    A  B  C")
for i, row_label in enumerate (["A","B","C"]): #Dispay the matrix
    print(f"{row_label}  {A_test[i]}")

print("\n Link Structure Verification")
for j, page in enumerate (["A","B","C"]):
    outlinks = np.where(A_test[:,j]== 1)[0]
    target_labels = [["A","B","C"][k] for k in outlinks]
    print (f"Page {page} links to {target_labels}")

3-page Web

Adjancency Matrix Visualization:
    A  B  C
A  [0. 0. 1.]
B  [1. 0. 0.]
C  [0. 1. 0.]

 Link Structure Verification
Page A links to ['B']
Page B links to ['C']
Page C links to ['A']


Now we will create the full web on which we will test the PageRank algorithm. We will model a 8-page web with:
* Assymetric Structure: Pages have different numbers of in/outlinks
* Spyder Trap: Pages F and G for an isolated cycle
* Dangling Node: Page H has no outgoing links

Graph Structure:

Main Cluster  (Pages A-E): <br>
A $\rightarrow$ B, C     (Page A links to B and C) <br>
B $\rightarrow$ C, D     (Page B links to C and D) <br>
C $\rightarrow$ A, D     (Page C links to A and D) <br>
D $\rightarrow$ A, E, H  (Page D links to A, E, and H) <br>
E $\rightarrow$ F        (Page E bridges to spider trap) <br>

Spider Trap (Pages F-G): <br>
F ↔ G        (F and G link to each other, isolated cycle)

Dangling Node: <br>
H → (none)   (Page H is a dead end - no outlinks)
```

**Visual representation:**
```
        ┌──→ B ───→ D ──┐
        │    ↓      ↑   │
    A ──┤    C ─────┘   ├──→ E ──→ F ↔ G
        │    ↑          │      (spider trap)
        └────┘          │
                        └──→ H (dangling)



In [103]:
#8-Page Complex Web


np.set_printoptions(precision=4, suppress=True) #floating numbers have 4 digits after .0 and no scientific format 10e-2

# Pages: A=0, B=1, C=2, D=3, E=4, F=5, G=6, H=7

A_web = np.array([
    #A,B,C,D,E,F,G,H
    [0,0,1,1,0,0,0,0], # A receives links from C, D
    [1,0,0,0,0,0,0,0], # B receives links from A
    [1,1,0,0,0,0,0,0], # C receives links from B, A
    [0,1,1,0,0,0,0,0], # D receives links from B, C
    [0,0,0,1,0,0,0,0], # E receives links from D
    [0,0,0,0,1,0,1,0], # F receives links from E, G
    [0,0,0,0,0,1,0,0], # G receives links from F (
    [0,0,0,1,0,0,0,0] # H receives links from D
])

A_web = np.array (A_web, dtype = 'float')

print("8-Page Web Adjacency Matrix:")
print("    ", "  ".join(page_labels))
for i, label in enumerate(page_labels):
    print(f"{label}   {A_web[i]}")

8-Page Web Adjacency Matrix:
     A  B  C  D  E  F  G  H
A   [0. 0. 1. 1. 0. 0. 0. 0.]
B   [1. 0. 0. 0. 0. 0. 0. 0.]
C   [1. 1. 0. 0. 0. 0. 0. 0.]
D   [0. 1. 1. 0. 0. 0. 0. 0.]
E   [0. 0. 0. 1. 0. 0. 0. 0.]
F   [0. 0. 0. 0. 1. 0. 1. 0.]
G   [0. 0. 0. 0. 0. 1. 0. 0.]
H   [0. 0. 0. 1. 0. 0. 0. 0.]


In [113]:
#Spider Traps & Dangling Nodes

#Page Labels
page_labels = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']

#Detailed Link Structure
for j, page in enumerate(page_labels):
    outlinks_indices = np.where(A_web[:,j]== 1)[0]
    if len(outlinks_indices)> 0 :
        outlinks_labels = [page_labels[k] for k in outlinks_indices]
        print(f"Page {page} -> {list(outlinks_labels)})")
    else:
        print(f"Page {page} -> None (dangling_node)\n")
        

#Identify dangling nodes (nodes without output links)
out_degrees = A_web.sum(axis=0)
dangling_nodes = np.where(out_degrees== 0)[0]
dangling_labels = [page_labels[i] for i in dangling_nodes]

print(f"Dangling Nodes:{dangling_labels} - no outgoing links\n")

Page A -> ['B', 'C'])
Page B -> ['C', 'D'])
Page C -> ['A', 'D'])
Page D -> ['A', 'E', 'H'])
Page E -> ['F'])
Page F -> ['G'])
Page G -> ['F'])
Page H -> None (dangling_node)

Dangling Nodes:['H'] - no outgoing links



## Transition Matrix

In this section we will create the Transition Matrix Conversion Function. The goal of this section is to transform the adjacency matrix into a column-stockastic transition matrix. In such matrix the sum of all columns is equal to 1. This function must be able to handle the dangling nodes in the web. 

Mathematical concept: If page $j$ has $k$ outlinks, each link gets probability $\frac{1}{k}$. For dangling nodes, we assign uniform probability to all pages.

The applied approach is desscribed below: 
1. First, we will convert convert an adjacency matrix to a column-stochastic transition matrix. Enty $M[i,j]$ is the probability of jumping from page $j$ to page $i$. In this step we will not assume the presence of dangling nodes (all columns have at least one 1).
2. Zero Division Handling - introduce dangling nodes. The applied approach in this step is to apply uniform distribution to all zero columns. This assigns an equal probability to come to the dangling node from any page.
3. Damping Factor - In this subsection we will    

In [125]:
### Column-Stochastic Matrix
out_degree = A_test.sum(axis=0) #calculate the sum of each column
M_test= A_test / out_degree #Normalize: divide each column by its sum -> making each column a probability distribution.

M_test

array([[0., 0., 1.],
       [1., 0., 0.],
       [0., 1., 0.]])

In [295]:
def build_transition_matrix_v1(adjacency_matrix):
    out_degree = adjacency_matrix.sum(axis=0)
    transition_matrix = adjacency_matrix / out_degree

    return transition_matrix

M_test_v1 = transition_matrix_v1(A_test)

#Print matrix
print(f"Result: Transition Matrix \n {M_test_v1}\n")

#Check taht columns sums to 1
column_sums = M_test_v1.sum(axis=0)
print(f"Column sumns: {column_sums}")
print (f"All columns sum to 1?: {np.allclose(column_sums,1.0)}")


Result: Transition Matrix 
 [[0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]]

Column sumns: [1. 1. 1.]
All columns sum to 1?: True


In [296]:
def build_transition_matrix_v2 (adjacent_matrix):
    mat_copy = adjacent_matrix.copy()  # Copy the Adjacenet Matrix
    page_count = mat_copy.shape[0] # count the number of pages

    out_degree = mat_copy.sum(axis = 0) # Find the column sum 
    dangling_nodes = (out_degree == 0) #Check if any column sum = 0

    if np.any(dangling_nodes):     
        mat_copy[:, dangling_nodes] = 1.0 / page_count # Set entire column to 1/n (uniform distribution)
        out_degree[dangling_nodes] = 1.0  

    tran_mat = mat_copy / out_degree

    return tran_mat

In [313]:
#Test to 3-page cycle (no-dangling nodes) 
m_test = build_transition_matrix_v2(A_test)

print("Transition Matrix m_test: \n")
print(f"{m_test} \n")
print(f" Column sums: {m_test.sum(axis=0)}")
print()

Transition Matrix m_test: 

[[0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]] 

 Column sums: [1. 1. 1.]



In [315]:
#Test to 8-page cycle (dangling nodes)
m_web_test = build_transition_matrix_v2(A_web)

print("Transition Matrix m_web_test:\n")
print(f"{m_web_test}\n")
print(f"Column sums: {m_web_test.sum(axis=0)}\n")

Transition Matrix m_web_test:

[[0.     0.     0.5    0.3333 0.     0.     0.     0.125 ]
 [0.5    0.     0.     0.     0.     0.     0.     0.125 ]
 [0.5    0.5    0.     0.     0.     0.     0.     0.125 ]
 [0.     0.5    0.5    0.     0.     0.     0.     0.125 ]
 [0.     0.     0.     0.3333 0.     0.     0.     0.125 ]
 [0.     0.     0.     0.     1.     0.     1.     0.125 ]
 [0.     0.     0.     0.     0.     1.     0.     0.125 ]
 [0.     0.     0.     0.3333 0.     0.     0.     0.125 ]]

Column sums: [1. 1. 1. 1. 1. 1. 1. 1.]



In [317]:
### Add Assertation to guarantee all column sum is exactly 1.0
def build_transition_matrix (adjacent_matrix):
    mat_copy = adjacent_matrix.copy()  # Copy the Adjacenet Matrix
    page_count = mat_copy.shape[0] # count the number of pages

    out_degree = mat_copy.sum(axis = 0) # Find the column sum 
    dangling_nodes = (out_degree == 0) #Check if any column sum = 0

    if np.any(dangling_nodes):     
        mat_copy[:, dangling_nodes] = 1.0 / page_count # Set entire column to 1/n (uniform distribution)
        out_degree[dangling_nodes] = 1.0  

    tran_mat = mat_copy / out_degree

    #Assert column-stochastic property
    assert np.allclose(m_web_test.sum(axis=0),1.0), "Matrix is not column stochastic"
    
    return tran_mat

In [3]:
### Teleportation Rule
### Damping Factor

## 

## 

## Power Iteration

In [ ]:
# Uniform Probability

In [ ]:
### L1 Norm Convergence

In [5]:
# Error Tracking and Visualizaiton

## Eigenvector Validaiton

In [ ]:
#Numpy Eigenvalue Solver

In [ ]:
#Normalizing Pricnipal Eigenvector

In [ ]:
# Visualizations

## Edge Case and Damping

In [6]:
# Damping Factor Formula

In [ ]:
# Stochastic Assertions

In [ ]:
# Final Importance Ranking